# Dimension accounts
1. read the silver accounts table
2. create the account surrogate key
3. select required columns
4. write the transformed data to gold dim_accounts table

In [0]:
#Imports
from pyspark.sql import Window
from pyspark.sql.functions import row_number,col

In [0]:
accounts_df = spark.read.table("neo_bank.bronze.accounts")
branches_df = spark.read.table("neo_bank.gold.dim_branches")
customers_df = spark.read.table("neo_bank.gold.dim_customers")


In [0]:
window_spec = Window.orderBy("account_id")
accounts_df = accounts_df.withColumn("account_sk",row_number().over(window_spec))

In [0]:
dim_accounts_df = (
    accounts_df.alias("a")
    .join(
        branches_df.alias("b"),
        col("a.branch_code")==col("b.branch_code"),
        "left"
    )
    .join(
        customers_df.alias("c"),
        col("a.customer_id")==col("c.customer_id"),
        "left"
    )
)

In [0]:
dim_accounts_df = dim_accounts_df.select(
    col("a.account_sk"),
    col("a.account_id"),
    col("c.customer_sk"),
    col("a.account_type"),
    col("a.balance"),
    col("a.currency"),
    col("b.branch_sk").alias("account_branch_sk"),
    col("a.status"),
    col("opened_date").alias("account_opened_date")
)

In [0]:
(
    dim_accounts_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","True")
        .saveAsTable("neo_bank.gold.dim_accounts")
)

In [0]:
%sql
select * from neo_bank.gold.dim_accounts